# Phase 2: RAG Query Only (Load Existing Vector DB)

This notebook:
1. Loads the BGE-large embedding model
2. Connects to existing Milvus vector database (`milvus_all_docs_mixed.db`)
3. Verifies collection exists and shows statistics
4. Initializes Gemini 2.5 Flash for answer generation
5. Provides RAG query interface for question answering

**Use this notebook when:**
- Your vector database is already created and populated
- You just want to perform RAG queries without re-processing documents
- You want fast startup without chunking/embedding overhead

**IMPORTANT - Before running this notebook:**
- This assumes you've already run `phase2_chunking_embedding_mixed.ipynb` to create the vector database
- ⚠️ **Database locking issue**: Milvus Lite (SQLite-based) only allows ONE connection at a time
- **If you see "zero vectors" or connection errors:**
  1. Shut down the kernel for `phase2_chunking_embedding_mixed.ipynb` (Kernel → Shut Down Kernel)
  2. Close any other notebooks that might have the database open
  3. Then restart this notebook
- Alternatively, if the other notebook is already running, just use Cell 14+ in that notebook directly

## Cell 1: Imports & GPU Check

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient
import google.generativeai as genai
import os

# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA not available, using CPU")

## Cell 2: Load BGE Embedding Model

In [ ]:
# Load BGE embedding model
model_name = "BAAI/bge-large-en-v1.5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")
embedding_model = SentenceTransformer(model_name, device=device)

# Test embedding to get dimension
test_embedding = embedding_model.encode(["test"])
embedding_dim = len(test_embedding[0])

print(f"\nModel loaded successfully!")
print(f"Model: {model_name}")
print(f"Device: {device}")
print(f"Embedding dimension: {embedding_dim}")
print(f"Max sequence length: {embedding_model.max_seq_length}")

## Cell 3: Configuration

In [ ]:
# Configuration
MILVUS_DB = "milvus_all_docs_mixed.db"
COLLECTION_NAME = "rag_all_docs_mixed"
EMBEDDING_DIM = 1024

print("Configuration:")
print(f"  Milvus DB: {MILVUS_DB}")
print(f"  Collection name: {COLLECTION_NAME}")
print(f"  Expected embedding dimension: {EMBEDDING_DIM}")

## Cell 4: Connect to Existing Milvus Database

In [ ]:
# Initialize Milvus client (connects to existing DB)
print(f"Connecting to existing Milvus database: {MILVUS_DB}")

try:
    client = MilvusClient(MILVUS_DB)
    
    # Verify collection exists
    if client.has_collection(COLLECTION_NAME):
        print(f"✓ Collection '{COLLECTION_NAME}' found!")
        
        # Get basic stats
        stats = client.get_collection_stats(COLLECTION_NAME)
        print(f"\nCollection Statistics:")
        print(f"  Total vectors: {stats['row_count']}")
        
        if stats['row_count'] == 0:
            print(f"\n⚠️  WARNING: Collection exists but has ZERO vectors!")
            print(f"  This usually means the database file is locked by another notebook.")
            print(f"  Please shut down the kernel for 'phase2_chunking_embedding_mixed.ipynb' and retry.")
        else:
            print(f"  Collection loaded and ready for queries!")
    else:
        print(f"✗ ERROR: Collection '{COLLECTION_NAME}' not found!")
        print(f"\nPlease run 'phase2_chunking_embedding_mixed.ipynb' first to create the vector database.")
        raise ValueError(f"Collection '{COLLECTION_NAME}' does not exist in '{MILVUS_DB}'")
        
except Exception as e:
    if "file has been opened by another program" in str(e) or "Open local milvus failed" in str(e):
        print(f"\n✗ ERROR: Database is locked by another program!")
        print(f"\n📋 TROUBLESHOOTING STEPS:")
        print(f"  1. In Jupyter, go to Kernel → Shut Down Kernel for 'phase2_chunking_embedding_mixed.ipynb'")
        print(f"  2. Close any other notebooks that might have this database open")
        print(f"  3. Then restart this notebook")
        print(f"\n💡 TIP: If you already have the other notebook running, just use Cell 14+ in that notebook directly!")
        raise
    else:
        print(f"\n✗ ERROR: {e}")
        raise

## Cell 5: Collection Statistics & Summary

In [ ]:
# Get detailed collection statistics
stats = client.get_collection_stats(COLLECTION_NAME)

# Sample a few records to show metadata structure
print(f"\n{'='*80}")
print(f"COLLECTION DETAILS")
print(f"{'='*80}")
print(f"\nCollection Name: {COLLECTION_NAME}")
print(f"Total Vectors: {stats['row_count']}")
print(f"Embedding Dimension: {EMBEDDING_DIM}")

# Query a sample record to show metadata fields
print(f"\nSample Record (showing metadata fields):")
sample_results = client.search(
    collection_name=COLLECTION_NAME,
    data=[[0.0] * EMBEDDING_DIM],  # dummy vector
    limit=1,
    output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"]
)

if sample_results and len(sample_results[0]) > 0:
    sample = sample_results[0][0]['entity']
    print(f"  Source File: {sample['source_file']}")
    print(f"  Doc Type: {sample['doc_type']}")
    print(f"  Chunker Used: {sample['chunker_used']}")
    print(f"  Chunk ID: {sample['chunk_id'] + 1}/{sample['total_chunks']}")
    print(f"  Text Preview: {sample['text'][:150]}...")

print(f"\n{'='*80}")
print(f"Database ready for RAG queries!")
print(f"{'='*80}")

## Cell 6: Initialize Gemini 2.5 Flash for Answer Generation

In [ ]:
# Configure Gemini API
# Note: Set your GEMINI_API_KEY or GOOGLE_API_KEY environment variable
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
model = 'gemini-2.5-flash'
if not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY not found in environment variables")
    print("Please set GEMINI_API_KEY or GOOGLE_API_KEY to use Gemini for answer generation")
    print("Example: export GEMINI_API_KEY='your-api-key-here'")
else:
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel(model)
    
    print("Gemini initialized successfully!")
    print(f"Model: {model}")
    print(f"Ready for RAG answer generation")

## Cell 7: RAG Function - Retrieve Context & Generate Answer

In [ ]:
def rag_query(query_text, top_k=5, show_full_chunks=True):
    """
    Complete RAG pipeline: Retrieve relevant chunks and generate answer using Gemini.
    
    Args:
        query_text: The question to answer
        top_k: Number of chunks to retrieve
        show_full_chunks: Whether to print full chunk text
    
    Returns:
        dict with answer, sources, and metadata
    """
    print(f"\n{'='*80}")
    print(f"QUERY: {query_text}")
    print(f"{'='*80}\n")
    
    # Step 1: Embed query
    print(f"Step 1: Embedding query...")
    query_embedding = embedding_model.encode([query_text])[0].tolist()
    
    # Step 2: Search vector database
    print(f"Step 2: Searching for top {top_k} relevant chunks...")
    search_results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"]
    )
    
    # Extract results
    context_texts = []
    sources = []
    
    print(f"\n{'='*80}")
    print(f"RETRIEVED CHUNKS:")
    print(f"{'='*80}\n")
    
    for idx, result in enumerate(search_results[0], 1):
        entity = result['entity']
        chunk_text = entity['text']
        context_texts.append(chunk_text)
        
        source_info = {
            'rank': idx,
            'source_file': entity['source_file'],
            'doc_type': entity['doc_type'],
            'chunk_id': entity['chunk_id'],
            'total_chunks': entity['total_chunks'],
            'chunker_used': entity['chunker_used'],
            'distance': result['distance'],
            'text': chunk_text
        }
        sources.append(source_info)
        
        # Display chunk info
        print(f"Chunk {idx}:")
        print(f"  Source: {entity['source_file']} ({entity['doc_type']})")
        print(f"  Chunker: {entity['chunker_used']}")
        print(f"  Chunk: {entity['chunk_id']+1}/{entity['total_chunks']}")
        print(f"  Distance: {result['distance']:.4f}")
        if show_full_chunks:
            print(f"  Text:\n{chunk_text}")
        else:
            print(f"  Text: {chunk_text[:200]}...")
        print(f"\n{'-'*80}\n")
    
    # Step 3: Build context for Gemini
    print(f"Step 3: Building context for LLM...")
    context = "\n\n".join([
        f"[Source {i+1}: {sources[i]['source_file']}]\n{text}" 
        for i, text in enumerate(context_texts)
    ])
    
    # Step 4: Generate answer with Gemini
    print(f"Step 4: Generating answer with Gemini 2.5 Flash...\n")
    
    prompt = f"""You are an expert financial analyst AI assistant with access to retrieved document context from a RAG (Retrieval-Augmented Generation) system.

IMPORTANT CONTEXT UNDERSTANDING:
The context provided below consists of relevant text chunks retrieved from financial documents (balance sheets, audit reports, financial statements, XBRL filings). These chunks may contain:
- Complete information directly answering the question
- Partial information requiring synthesis across multiple chunks
- Numerical data from tables requiring calculations
- Related information from which the answer can be derived through reasoning

YOUR CAPABILITIES:
1. **Multi-step reasoning**: You can derive answers through logical inference even if not explicitly stated
2. **Calculations**: You can perform arithmetic operations (addition, subtraction, multiplication, division, percentages, ratios, year-over-year changes)
3. **Cross-chunk synthesis**: Combine information from multiple retrieved chunks to form complete answers
4. **Financial domain knowledge**: Apply standard accounting principles and financial analysis techniques
5. **Pattern recognition**: Identify trends, anomalies, and relationships in financial data

RETRIEVED CONTEXT:
{context}

USER QUESTION: {query_text}

INSTRUCTIONS:
1. **Direct Information**: If the answer is explicitly stated in the context, provide it directly with source citations
2. **Derivable Information**: If the answer requires calculation or inference:
   - Clearly explain your reasoning steps
   - Show calculations with numbers from the context
   - Cite which sources provided the input data
3. **Incomplete Information**: If context provides partial information:
   - State what information is available
   - Explain what additional data would be needed
   - Provide the best possible answer with appropriate caveats
4. **No Information**: If context doesn't contain relevant information:
   - Explicitly state that the information is not available in the retrieved documents
   - Do NOT make up information or use knowledge outside the provided context

ANSWER FORMAT:
- Begin with a clear, concise answer
- Show calculations if performed (e.g., "Total Assets = 1,234,567 + 567,890 = 1,802,457")
- Cite sources: [Source 1], [Source 2], etc.
- If derived through reasoning, explain the logic
- Use proper financial terminology

Your response:"""
    
    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    # Display answer
    print(f"{'='*80}")
    print(f"GENERATED ANSWER:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n{'='*80}\n")
    
    return {
        'question': query_text,
        'answer': answer,
        'sources': sources,
        'num_chunks_retrieved': len(context_texts)
    }

print("RAG function ready!")
print("Usage: rag_query('your question here', top_k=5, show_full_chunks=True)")

In [ ]:
def rag_query_brute(query_text, top_k=5, show_full_chunks=True):
    """
    Complete RAG pipeline: Retrieve relevant chunks and generate answer using Gemini.
    
    Args:
        query_text: The question to answer
        top_k: Number of chunks to retrieve
        show_full_chunks: Whether to print full chunk text
    
    Returns:
        dict with answer, sources, and metadata
    """
    print(f"\n{'='*80}")
    print(f"QUERY: {query_text}")
    print(f"{'='*80}\n")
    
    # Step 1: Embed query
    print(f"Step 1: Embedding query...")
    query_embedding = embedding_model.encode([query_text])[0].tolist()
    
    # Step 2: Search vector database
    print(f"Step 2: Searching for top {top_k} relevant chunks...")
    search_results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"]
    )
    
    # Extract results
    context_texts = []
    sources = []
    
    print(f"\n{'='*80}")
    print(f"RETRIEVED CHUNKS:")
    print(f"{'='*80}\n")
    
    for idx, result in enumerate(search_results[0], 1):
        entity = result['entity']
        chunk_text = entity['text']
        context_texts.append(chunk_text)
        
        source_info = {
            'rank': idx,
            'source_file': entity['source_file'],
            'doc_type': entity['doc_type'],
            'chunk_id': entity['chunk_id'],
            'total_chunks': entity['total_chunks'],
            'chunker_used': entity['chunker_used'],
            'distance': result['distance'],
            'text': chunk_text
        }
        sources.append(source_info)
        
        # Display chunk info
        print(f"Chunk {idx}:")
        print(f"  Source: {entity['source_file']} ({entity['doc_type']})")
        print(f"  Chunker: {entity['chunker_used']}")
        print(f"  Chunk: {entity['chunk_id']+1}/{entity['total_chunks']}")
        print(f"  Distance: {result['distance']:.4f}")
        if show_full_chunks:
            print(f"  Text:\n{chunk_text}")
        else:
            print(f"  Text: {chunk_text[:200]}...")
        print(f"\n{'-'*80}\n")
    
    # Step 3: Build context for Gemini
    print(f"Step 3: Building context for LLM...")
    context = "\n\n".join([
        f"[Source {i+1}: {sources[i]['source_file']}]\n{text}" 
        for i, text in enumerate(context_texts)
    ])
    
    # Step 4: Generate answer with Gemini
    print(f"Step 4: Generating answer with Gemini 2.5 Flash...\n")
    
    prompt = f"""You are an expert financial analyst AI assistant, specialized in extracting precise information from financial documents retrieved via a RAG system. Apply rigorous analytical thinking.

    IMPORTANT CONTEXT UNDERSTANDING:
    The context provided below consists of relevant text chunks retrieved from various financial documents (e.g., balance sheets, audit reports, XBRL filings, MCA forms like AOC-4). These chunks may contain:
    - Explicit label-value pairs (e.g., "CIN: U85...") or selected options/checkboxes.
    - Narrative text describing financial or compliance details.
    - Data embedded within tables.
    - Partial information requiring synthesis.
    **PRIORITIZATION:** When available, prioritize information from explicitly labeled fields, selected options, or checkboxes over narrative descriptions for form-filling tasks.

    YOUR CAPABILITIES:
    1. **Multi-step reasoning**: Derive answers through logical inference.
    2. **Calculations**: Perform necessary arithmetic.
    3. **Cross-chunk synthesis**: Combine info from multiple chunks.
    4. **Financial domain knowledge**: Apply standard accounting & compliance principles.
    5. **Pattern recognition**: Identify relevant data structures (label-value, selections).
    6. **Batch processing**: Answer multiple questions systematically.

    RETRIEVED CONTEXT:
    --- START CONTEXT ---
    {context}
    --- END CONTEXT ---

    USER QUESTIONS (BATCH - for AOC-4 form):
    {query_text}

    TASK:
    Analyze the context to answer ALL user questions accurately for filling the AOC-4 form. Structure your response clearly by grouping related questions.

    ANSWER GUIDELINES:

    **For each question:**
    - **Found**: If the *exact* answer is present (especially in a labeled field or as a selection), provide it precisely. Extract values **exactly** as they appear in the source if possible (e.g., '5000000.0', not '5,000,000'). Cite specific source(s) like `[Source X]`. If chunk metadata is available (e.g., filename), include it: `[Source X: filename.pdf, chunk Y]`.
    - **Derived**: If the answer requires calculation, synthesis, or inference based *only* on the provided context, show brief reasoning (e.g., "Calculated from Shares * Value per Share") and cite all contributing sources.
    - **Not Found**: If the information is genuinely absent in *all* retrieved chunks, state: "Not found in retrieved documents".
    - **Conflicting**: If different sources provide conflicting answers for the same fact, state **both conflicting answers and their sources clearly**. Example: "Source A indicates 'Yes', while Source B indicates 'No'".

    **Formatting & Specific Question Types:**
    - Use clear headings/numbering matching the question numbers. Group related questions logically under `## SECTION` headings.
    - **Yes/No/Selection Questions:**
        - Look for explicit indicators ('Yes'/'No' selected, checked boxes, specific options like 'Adopted Financial Statements').
        - If found, answer **"Yes"** or **"No"**.
        - Use **"Not Applicable"** *only* if the question itself is irrelevant based on other facts (e.g., CAG questions for a non-government company clearly identified as private). Do *not* use "Not Applicable" if the form simply indicates "No".
        - If no explicit selection is found, state **"Not Found"**.
    - **Fill-in-the-blank Questions:** Provide the specific value/text or "Not Found".
    - **Date Questions:** Use DD/MM/YYYY format. **Crucially**, for dates like AGM, prioritize the date relevant to the *current* financial reporting period (ending >FinYearEndDate< - *you might need to inject this date here*). Be wary of dates mentioned for *past* events within the context. Extract the date *exactly* if possible before reformatting.
    - Keep answers concise, extracting only the required piece of information.

    **Special Instructions:**
    - **Accuracy First**: Never guess or fabricate. If unsure, state "Not Found".
    - **Data Quality**: Briefly note any obvious data quality issues observed (e.g., the Secretarial Audit discrepancy noted previously).

    **Output Structure:**
    Organize your response as:

    ## SECTION 1: [Logical Group Name, e.g., Company Identification]
    1.  **Question text**
        **Answer:** [Your answer]
        **Source:** [Source citations]

    2.  **Question text**
        **Answer:** [Your answer]
        **Source:** [Source citations]
    ---
    ## SECTION 2: [Logical Group Name, e.g., Financial Reporting Period & Approval]
    ...

    Begin your structured response:"""
    
    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    # Display answer
    print(f"{'='*80}")
    print(f"GENERATED ANSWER:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n{'='*80}\n")
    
    return {
        'question': query_text,
        'answer': answer,
        'sources': sources,
        'num_chunks_retrieved': len(context_texts)
    }

print("RAG function ready!")
print("Usage: rag_query_brute('your question here', top_k=5, show_full_chunks=True)")

In [ ]:
all_queries = '''
            Language (English/Hindi)
            Authorised capital of the company as on the date of filling (in Rs.)
            1.(a) Corporate Identity Number (CIN) of company
            2.(a) Name of the company
            2.(b) Address of the registered office of the company
            2.(c) e-mail ID of the company
            3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)
            4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)
            4.(b) Nature of financial statements: ____? 4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)
            5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 5.(c) Due date of AGM:____ (DD/MM/YYYY). 5.(d) Whether any extension for financial year or AGM granted (Yes/No)
            6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)
            7. Type of Industry
            8. Whether consolidated Financial State,emts are also being filed (Yes/No)
            9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)
            9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)
            10. Whether Secretarial Audit is applicable (Yes/No)
            11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)
            '''

## Cell 8: Test RAG - Example Query

In [ ]:
# Example query to test the RAG system
result = rag_query_brute(
    all_queries,
    top_k=350,
    show_full_chunks=False  # Set to True to see full chunk text
)

## Cell 9: Custom Query Interface

Use this cell to run your own custom queries. You can:
- Ask single questions
- Process a list of questions
- Adjust `top_k` to retrieve more/fewer chunks
- Toggle `show_full_chunks` to control output verbosity

In [ ]:
# Single query example
custom_result = rag_query(
    "What is the authorized capital of the company",
    top_k=5,
    show_full_chunks=True
)

In [ ]:
# Multiple queries example (uncomment to use)
"""
questions = [
    "What is the Corporate Identity Number (CIN) of the company?",
    "What is the name of the company?",
    "What is the address of the registered office?",
    "What is the email ID of the company?",
    "What is the financial year covered in the statements?",
    "What is the authorized capital of the company?",
    "Show me balance sheet information for ExampleCo",
    "What are the revenue figures for FY 23-24?"
]

results = []
for question in questions:
    result = rag_query(question, top_k=5, show_full_chunks=False)
    results.append(result)
    print("\n" + "="*80 + "\n")
"""
pass

In [ ]:
def rag_query_with_hyde(query_text, top_k=5, show_full_chunks=True, show_hypothetical=True):
    """
    RAG pipeline with HyDE (Hypothetical Document Embeddings).
    Generates a hypothetical answer first, embeds it, then retrieves relevant chunks.
    
    Args:
        query_text: The question to answer
        top_k: Number of chunks to retrieve
        show_full_chunks: Whether to print full chunk text
        show_hypothetical: Whether to print the generated hypothetical document
    
    Returns:
        dict with answer, sources, hypothetical_doc, and metadata
    """
    print(f"\n{'='*80}")
    print(f"QUERY (with HyDE): {query_text}")
    print(f"{'='*80}\n")
    
    # Step 1: Generate hypothetical document using Gemini
    print(f"Step 1: Generating hypothetical document...")
    
    hyde_prompt = f"""You are a financial document expert. Given a question, write a hypothetical passage from a financial document that would perfectly answer this question.

The passage should:
- Be written in formal financial document style
- Contain specific numbers, dates, and financial terminology where relevant
- Be comprehensive (2-3 sentences)
- Sound like it came from an actual financial statement or audit report

Question: {query_text}

Write the hypothetical document passage:"""
    
    try:
        hyde_response = gemini_model.generate_content(hyde_prompt)
        hypothetical_doc = hyde_response.text.strip()
    except Exception as e:
        print(f"Error generating hypothetical document: {e}")
        print("Falling back to standard query embedding...")
        hypothetical_doc = query_text
    
    if show_hypothetical:
        print(f"\nGenerated Hypothetical Document:")
        print(f"{'-'*80}")
        print(f"{hypothetical_doc}")
        print(f"{'-'*80}\n")
    
    # Step 2: Embed the hypothetical document (not the original query)
    print(f"Step 2: Embedding hypothetical document...")
    hyde_embedding = embedding_model.encode([hypothetical_doc])[0].tolist()
    
    # Step 3: Search vector database using HyDE embedding
    print(f"Step 3: Searching for top {top_k} relevant chunks using HyDE embedding...")
    search_results = client.search(
        collection_name=COLLECTION_NAME,
        data=[hyde_embedding],
        limit=top_k,
        output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"]
    )
    
    # Extract results
    context_texts = []
    sources = []
    
    print(f"\n{'='*80}")
    print(f"RETRIEVED CHUNKS (via HyDE):")
    print(f"{'='*80}\n")
    
    for idx, result in enumerate(search_results[0], 1):
        entity = result['entity']
        chunk_text = entity['text']
        context_texts.append(chunk_text)
        
        source_info = {
            'rank': idx,
            'source_file': entity['source_file'],
            'doc_type': entity['doc_type'],
            'chunk_id': entity['chunk_id'],
            'total_chunks': entity['total_chunks'],
            'chunker_used': entity['chunker_used'],
            'distance': result['distance'],
            'text': chunk_text
        }
        sources.append(source_info)
        
        # Display chunk info
        print(f"Chunk {idx}:")
        print(f"  Source: {entity['source_file']} ({entity['doc_type']})")
        print(f"  Chunker: {entity['chunker_used']}")
        print(f"  Chunk: {entity['chunk_id']+1}/{entity['total_chunks']}")
        print(f"  Distance: {result['distance']:.4f}")
        if show_full_chunks:
            print(f"  Text:\n{chunk_text}")
        else:
            print(f"  Text: {chunk_text[:200]}...")
        print(f"\n{'-'*80}\n")
    
    # Step 4: Build context for Gemini
    print(f"Step 4: Building context for LLM...")
    context = "\n\n".join([
        f"[Source {i+1}: {sources[i]['source_file']}]\n{text}" 
        for i, text in enumerate(context_texts)
    ])
    
    # Step 5: Generate final answer with Gemini using ORIGINAL query
    print(f"Step 5: Generating final answer with Gemini 2.5 Flash...\n")
    
    prompt = f"""You are an expert financial analyst AI assistant with access to retrieved document context from a RAG (Retrieval-Augmented Generation) system.

IMPORTANT CONTEXT UNDERSTANDING:
The context provided below consists of relevant text chunks retrieved from financial documents (balance sheets, audit reports, financial statements, XBRL filings). These chunks may contain:
- Complete information directly answering the question
- Partial information requiring synthesis across multiple chunks
- Numerical data from tables requiring calculations
- Related information from which the answer can be derived through reasoning

YOUR CAPABILITIES:
1. **Multi-step reasoning**: You can derive answers through logical inference even if not explicitly stated
2. **Calculations**: You can perform arithmetic operations (addition, subtraction, multiplication, division, percentages, ratios, year-over-year changes)
3. **Cross-chunk synthesis**: Combine information from multiple retrieved chunks to form complete answers
4. **Financial domain knowledge**: Apply standard accounting principles and financial analysis techniques
5. **Pattern recognition**: Identify trends, anomalies, and relationships in financial data

RETRIEVED CONTEXT:
{context}

USER QUESTION: {query_text}

INSTRUCTIONS:
1. **Direct Information**: If the answer is explicitly stated in the context, provide it directly with source citations
2. **Derivable Information**: If the answer requires calculation or inference:
   - Clearly explain your reasoning steps
   - Show calculations with numbers from the context
   - Cite which sources provided the input data
3. **Incomplete Information**: If context provides partial information:
   - State what information is available
   - Explain what additional data would be needed
   - Provide the best possible answer with appropriate caveats
4. **No Information**: If context doesn't contain relevant information:
   - Explicitly state that the information is not available in the retrieved documents
   - Do NOT make up information or use knowledge outside the provided context

ANSWER FORMAT:
- Begin with a clear, concise answer
- Show calculations if performed (e.g., "Total Assets = 1,234,567 + 567,890 = 1,802,457")
- Cite sources: [Source 1], [Source 2], etc.
- If derived through reasoning, explain the logic
- Use proper financial terminology

Your response:"""
    
    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    # Display answer
    print(f"{'='*80}")
    print(f"GENERATED ANSWER:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n{'='*80}\n")
    
    return {
        'question': query_text,
        'answer': answer,
        'hypothetical_doc': hypothetical_doc,
        'sources': sources,
        'num_chunks_retrieved': len(context_texts),
        'method': 'HyDE'
    }

print("HyDE-enhanced RAG function ready!")
print("Usage: rag_query_with_hyde('your question here', top_k=5, show_full_chunks=True)")


In [ ]:
# Compare standard vs HyDE retrieval
result_hyde = rag_query_with_hyde(
    "Authorised capital of the company",
    top_k=5,
    show_full_chunks=False
)

In [ ]:
def compare_retrieval(query_text, top_k=5):
    """Compare standard retrieval vs HyDE retrieval"""
    print("\n" + "="*80)
    print("COMPARISON: Standard vs HyDE Retrieval")
    print("="*80)
    
    # Standard retrieval
    print("\n[1] STANDARD RETRIEVAL:")
    result_standard = rag_query(query_text, top_k=top_k, show_full_chunks=False)
    
    # HyDE retrieval
    print("\n[2] HYDE RETRIEVAL:")
    result_hyde = rag_query_with_hyde(query_text, top_k=top_k, show_full_chunks=False)
    
    # Compare sources
    print("\n" + "="*80)
    print("SOURCE COMPARISON:")
    print("="*80)
    
    standard_sources = set([s['source_file'] for s in result_standard['sources']])
    hyde_sources = set([s['source_file'] for s in result_hyde['sources']])
    
    print(f"\nStandard sources: {standard_sources}")
    print(f"HyDE sources: {hyde_sources}")
    print(f"\nUnique to Standard: {standard_sources - hyde_sources}")
    print(f"Unique to HyDE: {hyde_sources - standard_sources}")
    print(f"Common: {standard_sources & hyde_sources}")
    
    return {
        'standard': result_standard,
        'hyde': result_hyde
    }



In [ ]:
comparison = compare_retrieval("11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)", top_k=20)


In [ ]:
queries = [
            #Page 1 questions
            "Language (English/Hindi)"
            "Authorised capital of the company as on the date of filling (in Rs.)",
            "1.(a) Corporate Identity Number (CIN) of company",
            "2.(a) Name of the company",
            "2.(b) Address of the registered office of the company",
            "2.(c) e-mail ID of the company",
            "3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)",
            "4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)",
            "4.(b) Nature of financial statements: ____? 4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)",
            "5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 5.(c) Due date of AGM:____ (DD/MM/YYYY). 5.(d) Whether any extension for financial year or AGM granted (Yes/No)",
            "6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)",

            #Page 2 questions
            "7. Type of Industry",
            "8. Whether consolidated Financial State,emts are also being filed (Yes/No)",
            "9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)",
            "9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)",
            "10. Whether Secretarial Audit is applicable (Yes/No)",
            "11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)"
    ]

In [ ]:
for query in queries:
    result_hyde = rag_query_with_hyde(
    query,
    top_k=100,
    show_full_chunks=False
)

In [ ]:
all_queries = [
            "Language (English/Hindi)"
            "Authorised capital of the company as on the date of filling (in Rs.)"
            "1.(a) Corporate Identity Number (CIN) of company"
            "2.(a) Name of the company"
            "2.(b) Address of the registered office of the company"
            "2.(c) e-mail ID of the company"
            "3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)"
            "4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)"
            "4.(b) Nature of financial statements: ____? 4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)"
            "5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 5.(c) Due date of AGM:____ (DD/MM/YYYY). 5.(d) Whether any extension for financial year or AGM granted (Yes/No)"
            "6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)"

            #Page 2 questions
            "7. Type of Industry"
            "8. Whether consolidated Financial State,emts are also being filed (Yes/No)"
            "9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)"
            "9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)"
            "10. Whether Secretarial Audit is applicable (Yes/No)"
            "11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)"
            ]
    

In [ ]:
result = rag_query(
    all_queries,
    top_k=150,
    show_full_chunks=False  # Set to True to see full chunk text
)

In [ ]:
def batch_rag_query(questions_text, top_k_per_question=5, show_retrieval_progress=True):
    """
    Query vector DB with multiple questions, collect unique chunks, then generate answers.
    
    Args:
        questions_text: String containing all questions (line-separated)
        top_k_per_question: Number of chunks to retrieve per question
        show_retrieval_progress: Whether to show retrieval details
    
    Returns:
        dict with answer, all_sources, and metadata
    """
    # Parse questions into list
    questions = [q.strip() for q in questions_text.strip().split('\n') if q.strip()]
    
    print(f"\n{'='*80}")
    print(f"BATCH RAG QUERY - {len(questions)} Questions")
    print(f"{'='*80}\n")
    
    # Step 1: Retrieve chunks for each question
    print(f"Step 1: Retrieving top {top_k_per_question} chunks per question...")
    print(f"         This will query {len(questions)} questions\n")
    
    # Use a dictionary to track unique chunks (keyed by chunk ID)
    unique_chunks = {}
    question_chunk_mapping = {}  # Track which questions retrieved which chunks
    
    for idx, question in enumerate(questions, 1):
        if show_retrieval_progress:
            print(f"[{idx}/{len(questions)}] {question[:60]}...")
        
        # Embed and search
        query_embedding = embedding_model.encode([question])[0].tolist()
        search_results = client.search(
            collection_name=COLLECTION_NAME,
            data=[query_embedding],
            limit=top_k_per_question,
            output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"]
        )
        
        # Collect unique chunks
        question_chunks = []
        for result in search_results[0]:
            entity = result['entity']
            # Create unique ID for chunk (using source_file + chunk_id)
            chunk_key = f"{entity['source_file']}_{entity['chunk_id']}"
            
            if chunk_key not in unique_chunks:
                unique_chunks[chunk_key] = {
                    'text': entity['text'],
                    'source_file': entity['source_file'],
                    'doc_type': entity['doc_type'],
                    'chunk_id': entity['chunk_id'],
                    'total_chunks': entity['total_chunks'],
                    'chunker_used': entity['chunker_used'],
                    'distance': result['distance'],
                    'retrieved_by_questions': []
                }
            
            # Track which questions retrieved this chunk
            unique_chunks[chunk_key]['retrieved_by_questions'].append(idx)
            question_chunks.append(chunk_key)
        
        question_chunk_mapping[idx] = question_chunks
        
        if show_retrieval_progress:
            print(f"         Retrieved {len(question_chunks)} chunks, {len(unique_chunks)} unique total")
    
    print(f"\n{'='*80}")
    print(f"RETRIEVAL SUMMARY:")
    print(f"{'='*80}")
    print(f"Total questions queried: {len(questions)}")
    print(f"Chunks retrieved per question: {top_k_per_question}")
    print(f"Total retrievals: {len(questions) * top_k_per_question}")
    print(f"Unique chunks collected: {len(unique_chunks)}")
    print(f"Deduplication saved: {(len(questions) * top_k_per_question) - len(unique_chunks)} redundant chunks")
    
    # Step 2: Build combined context
    print(f"\nStep 2: Building combined context from {len(unique_chunks)} unique chunks...")
    
    # Sort chunks by how many questions retrieved them (most relevant first)
    sorted_chunks = sorted(
        unique_chunks.values(),
        key=lambda x: (len(x['retrieved_by_questions']), -x['distance']),
        reverse=True
    )
    
    context = "\n\n".join([
        f"[Source {i+1}: {chunk['source_file']} - Chunk {chunk['chunk_id']+1}/{chunk['total_chunks']}]\n{chunk['text']}"
        for i, chunk in enumerate(sorted_chunks)
    ])
    
    # Step 3: Generate answer with Gemini
    print(f"Step 3: Generating comprehensive answer with Gemini 2.5 Flash...\n")
    
    prompt = f"""You are an expert financial analyst AI assistant with access to retrieved document context from a RAG (Retrieval-Augmented Generation) system.

IMPORTANT CONTEXT UNDERSTANDING:
The context provided below consists of relevant text chunks retrieved from financial documents (balance sheets, audit reports, financial statements, XBRL filings, MCA forms). These chunks may contain:
- Complete information directly answering the questions
- Partial information requiring synthesis across multiple chunks
- Numerical data from tables requiring calculations
- Related information from which answers can be derived through reasoning

YOUR CAPABILITIES:
1. **Multi-step reasoning**: Derive answers through logical inference even if not explicitly stated
2. **Calculations**: Perform arithmetic operations (addition, subtraction, multiplication, division, percentages, ratios, year-over-year changes)
3. **Cross-chunk synthesis**: Combine information from multiple retrieved chunks to form complete answers
4. **Financial domain knowledge**: Apply standard accounting principles and financial analysis techniques
5. **Pattern recognition**: Identify trends, anomalies, and relationships in financial data
6. **Batch processing**: Answer multiple related questions systematically by creating logical groupings

RETRIEVED CONTEXT ({len(unique_chunks)} unique sources):
{context}

USER QUESTIONS (BATCH):
{questions_text}

TASK:
You are given a list of related questions about a company's financial filings and regulatory requirements. Your task is to:

1. **Analyze all questions** and identify logical groupings (e.g., company information, financial year details, meeting dates, compliance requirements)
2. **Search the context systematically** for information relevant to each question
3. **Answer ALL questions** in a structured format
4. **Create clear sections** to organize your responses

ANSWER GUIDELINES:

**For each question:**
- **Found**: If information is in the context, provide the exact answer with [Source X] citation
- **Derived**: If requires calculation/inference, show your work and cite sources
- **Not Found**: If information is unavailable, state: "Not found in retrieved documents"
- **Partial**: If only partial information exists, provide what's available and note what's missing

**Formatting Requirements:**
- Use clear numbering matching the question numbers (1.a, 2.a, etc.)
- For yes/no questions, provide: "Yes" or "No" or "Not specified" 
- For fill-in-the-blank questions, provide the specific value or "Not found"
- For date questions, use the format requested (DD/MM/YYYY)
- Keep answers concise but complete
- Group related questions when logical (e.g., AGM-related questions together)

**Special Instructions:**
- Prioritize accuracy over completeness - never fabricate information
- If multiple sources conflict, note the discrepancy
- For regulatory/compliance questions, be precise with Yes/No/Not Applicable
- Highlight any data quality issues (e.g., inconsistent dates, missing mandatory fields)

**Output Structure:**
Organize your response with clear numbering or sections. For each question provide:
- The answer (direct value, Yes/No, or explanation)
- Source citations when applicable

Begin your structured response:"""
    
    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    # Display answer
    print(f"{'='*80}")
    print(f"GENERATED ANSWER:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n{'='*80}\n")
    
    return {
        'questions': questions,
        'num_questions': len(questions),
        'answer': answer,
        'unique_chunks': len(unique_chunks),
        'total_retrievals': len(questions) * top_k_per_question,
        'all_sources': sorted_chunks,
        'question_chunk_mapping': question_chunk_mapping
    }

print("Batch RAG function ready!")
print("Usage: batch_rag_query(all_queries, top_k_per_question=5)")


In [ ]:
all_queries = '''
Language (English/Hindi)
Authorised capital of the company as on the date of filling (in Rs.)
1.(a) Corporate Identity Number (CIN) of company
2.(a) Name of the company
2.(b) Address of the registered office of the company
2.(c) e-mail ID of the company
3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)
4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)
4.(b) Nature of financial statements: ____? 
4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 
4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)
5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 
5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 
5.(c) Due date of AGM:____ (DD/MM/YYYY). 
5.(d) Whether any extension for financial year or AGM granted (Yes/No)
6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 
6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)
7. Type of Industry
8. Whether consolidated Financial State,emts are also being filed (Yes/No)
9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)
9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)
10. Whether Secretarial Audit is applicable (Yes/No)
11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)
'''

result = batch_rag_query(all_queries, top_k_per_question=5)

In [ ]:
def batch_rag_query1(questions_text, genqs, top_k_per_question=5, show_retrieval_progress=True, exclude_excel=True):
    """
    Query vector DB with multiple questions, collect unique chunks, then generate answers.
    
    Args:
        questions_text: String containing all questions (line-separated)
        top_k_per_question: Number of chunks to retrieve per question
        show_retrieval_progress: Whether to show retrieval details
        exclude_excel: If True, excludes XLSX chunks from retrieval (default: True)
    
    Returns:
        dict with answer, all_sources, and metadata
    """
    # Parse questions into list
    questions = [q.strip() for q in questions_text.strip().split('\n') if q.strip()]
    
    print(f"\n{'='*80}")
    print(f"BATCH RAG QUERY - {len(questions)} Questions")
    if exclude_excel:
        print(f"Note: Excluding Excel (XLSX) chunks from retrieval")
    print(f"{'='*80}\n")
    
    # Step 1: Retrieve chunks for each question
    print(f"Step 1: Retrieving top {top_k_per_question} chunks per question...")
    print(f"         This will query {len(questions)} questions\n")
    
    # Use a dictionary to track unique chunks (keyed by chunk ID)
    unique_chunks = {}
    question_chunk_mapping = {}  # Track which questions retrieved which chunks
    
    for idx, question in enumerate(questions, 1):
        if show_retrieval_progress:
            print(f"[{idx}/{len(questions)}] {question[:60]}...")
        
        # Embed and search
        query_embedding = embedding_model.encode([question])[0].tolist()
        
        # Build search parameters based on exclude_excel
        if exclude_excel:
            search_results = client.search(
                collection_name=COLLECTION_NAME,
                data=[query_embedding],
                limit=top_k_per_question,
                output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"],
                filter='doc_type != "xlsx"'  # Using 'filter' parameter instead of 'expr'
            )
        else:
            search_results = client.search(
                collection_name=COLLECTION_NAME,
                data=[query_embedding],
                limit=top_k_per_question,
                output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks", "chunker_used"]
            )
        
        # Collect unique chunks
        question_chunks = []
        for result in search_results[0]:
            entity = result['entity']
            # Create unique ID for chunk (using source_file + chunk_id)
            chunk_key = f"{entity['source_file']}_{entity['chunk_id']}"
            
            if chunk_key not in unique_chunks:
                unique_chunks[chunk_key] = {
                    'text': entity['text'],
                    'source_file': entity['source_file'],
                    'doc_type': entity['doc_type'],
                    'chunk_id': entity['chunk_id'],
                    'total_chunks': entity['total_chunks'],
                    'chunker_used': entity['chunker_used'],
                    'distance': result['distance'],
                    'retrieved_by_questions': []
                }
            
            # Track which questions retrieved this chunk
            unique_chunks[chunk_key]['retrieved_by_questions'].append(idx)
            question_chunks.append(chunk_key)
        
        question_chunk_mapping[idx] = question_chunks
        
        if show_retrieval_progress:
            print(f"         Retrieved {len(question_chunks)} chunks, {len(unique_chunks)} unique total")
    
    print(f"\n{'='*80}")
    print(f"RETRIEVAL SUMMARY:")
    print(f"{'='*80}")
    print(f"Total questions queried: {len(questions)}")
    print(f"Chunks retrieved per question: {top_k_per_question}")
    print(f"Total retrievals: {len(questions) * top_k_per_question}")
    print(f"Unique chunks collected: {len(unique_chunks)}")
    print(f"Deduplication saved: {(len(questions) * top_k_per_question) - len(unique_chunks)} redundant chunks")
    if exclude_excel:
        print(f"Excel chunks excluded: Yes (only PDF, DOCX, JPG, etc.)")
    
    # Show doc type breakdown
    doc_types = {}
    for chunk in unique_chunks.values():
        doc_type = chunk['doc_type']
        doc_types[doc_type] = doc_types.get(doc_type, 0) + 1
    print(f"\nChunks by document type:")
    for doc_type, count in sorted(doc_types.items()):
        print(f"  {doc_type}: {count} chunks")
    
    # Step 2: Build combined context
    print(f"\nStep 2: Building combined context from {len(unique_chunks)} unique chunks...")
    
    # Sort chunks by how many questions retrieved them (most relevant first)
    sorted_chunks = sorted(
        unique_chunks.values(),
        key=lambda x: (len(x['retrieved_by_questions']), -x['distance']),
        reverse=True
    )
    
    context = "\n\n".join([
        f"[Source {i+1}: {chunk['source_file']} - Chunk {chunk['chunk_id']+1}/{chunk['total_chunks']}]\n{chunk['text']}"
        for i, chunk in enumerate(sorted_chunks)
    ])
    
    # Step 3: Generate answer with Gemini
    print(f"Step 3: Generating comprehensive answer with Gemini 2.5 Flash...\n")
    
    prompt = f"""You are an expert financial analyst AI assistant with access to retrieved document context from a RAG (Retrieval-Augmented Generation) system.

IMPORTANT CONTEXT UNDERSTANDING:
The context provided below consists of relevant text chunks retrieved from financial documents (balance sheets, audit reports, financial statements, XBRL filings, MCA forms). These chunks may contain:
- Complete information directly answering the questions
- Partial information requiring synthesis across multiple chunks
- Numerical data from tables requiring calculations
- Related information from which answers can be derived through reasoning

YOUR CAPABILITIES:
1. **Multi-step reasoning**: Derive answers through logical inference even if not explicitly stated
2. **Calculations**: Perform arithmetic operations (addition, subtraction, multiplication, division, percentages, ratios, year-over-year changes)
3. **Cross-chunk synthesis**: Combine information from multiple retrieved chunks to form complete answers
4. **Financial domain knowledge**: Apply standard accounting principles and financial analysis techniques
5. **Pattern recognition**: Identify trends, anomalies, and relationships in financial data
6. **Batch processing**: Answer multiple related questions systematically by creating logical groupings

RETRIEVED CONTEXT ({len(unique_chunks)} unique sources):
{context}

USER QUESTIONS (BATCH):
{genqs}

TASK:
You are given a list of related questions about a company's financial filings and regulatory requirements. Your task is to:

1. **Analyze all questions** and identify logical groupings (e.g., company information, financial year details, meeting dates, compliance requirements)
2. **Search the context systematically** for information relevant to each question
3. **Answer ALL questions** in a structured format
4. **Create clear sections** to organize your responses

ANSWER GUIDELINES:

**For each question:**
- **Found**: If information is in the context, provide the exact answer with [Source X] citation
- **Derived**: If requires calculation/inference, show your work and cite sources
- **Not Found**: If information is unavailable, state: "Not found in retrieved documents"
- **Partial**: If only partial information exists, provide what's available and note what's missing

**Formatting Requirements:**
- Use clear numbering matching the question numbers (1.a, 2.a, etc.)
- For yes/no questions, provide: "Yes" or "No" or "Not specified" 
- For fill-in-the-blank questions, provide the specific value or "Not found"
- For date questions, use the format requested (DD/MM/YYYY)
- Keep answers concise but complete
- Group related questions when logical (e.g., AGM-related questions together)

**Special Instructions:**
- Prioritize accuracy over completeness - never fabricate information
- If multiple sources conflict, note the discrepancy
- For regulatory/compliance questions, be precise with Yes/No/Not Applicable
- Highlight any data quality issues (e.g., inconsistent dates, missing mandatory fields)

**Output Structure:**
Organize your response with clear numbering or sections. For each question provide:
- The answer (direct value, Yes/No, or explanation)
- Source citations when applicable

Begin your structured response:"""
    
    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    # Display answer
    print(f"{'='*80}")
    print(f"GENERATED ANSWER:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n{'='*80}\n")
    
    return {
        'questions': questions,
        'num_questions': len(questions),
        'answer': answer,
        'unique_chunks': len(unique_chunks),
        'total_retrievals': len(questions) * top_k_per_question,
        'all_sources': sorted_chunks,
        'question_chunk_mapping': question_chunk_mapping,
        'doc_type_breakdown': doc_types,
        'excel_excluded': exclude_excel
    }

print("Batch RAG function ready!")
print("Usage: batch_rag_query1(all_queries, top_k_per_question=5, exclude_excel=True)")


In [ ]:
all_queries_ret = '''
Language
Authorised capital of the company as on the date of filling (in Rs.)
Corporate Identity Number (CIN) of company
Name of the company
Address of the registered office of the company
e-mail ID of the company
Financial year to which financial statement relates
Date of Board of Director's meeting in which financial statements are approved
Nature of financial statements
Whether provisional financial statements filed earlier
Whether adopted in adjourned AGM
Whether Annual General Meeting (AGM) held
Date of AGM
Due date of AGM
Whether any extension for financial year or AGM granted
Whether Schedule 3 of the Companies Act, 2013 is applicable
Basis of financial statements (AS Taxonomy or Ind-AS Taxonomy)
Type of Industry
Whether consolidated Financial Statements are also being filed
Whether CAG of India has commented upon or supplemented the audit report under section 143
Whether CAG of India has conducted supplementary or test audit under section 143
Whether Secretarial Audit is applicable
Whether detailed disclosure with respect to Director's report Sec 134(3) is attached
'''
all_queries_gen = '''
Language (English/Hindi)
Authorised capital of the company as on the date of filling (in Rs.)
1.(a) Corporate Identity Number (CIN) of company
2.(a) Name of the company
2.(b) Address of the registered office of the company
2.(c) e-mail ID of the company
3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)
4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)
4.(b) Nature of financial statements: ____? 
4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 
4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)
5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 
5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 
5.(c) Due date of AGM:____ (DD/MM/YYYY). 
5.(d) Whether any extension for financial year or AGM granted (Yes/No)
6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 
6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)
7. Type of Industry
8. Whether consolidated Financial State,emts are also being filed (Yes/No)
9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)
9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)
10. Whether Secretarial Audit is applicable (Yes/No)
11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)
'''

batch_rag_query1(all_queries_gen, all_queries_gen, top_k_per_question=8, exclude_excel=True)